# Ames Housing Price Prediction
MDI3003 – Advanced Predictive Analytics

**Student Name:** Harshita Bogineni

**Reg No:** 23MID0043

In [19]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
SEED=42


## Load Ames Housing Dataset

In [20]:
# Download 'train.csv' from Kaggle House Prices
df=pd.read_csv('train.csv')
display(df.head())
print(df.shape)

   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape LandContour Utilities LotConfig LandSlope Neighborhood Condition1 Condition2 BldgType HouseStyle  OverallQual  OverallCond  YearBuilt  YearRemodAdd RoofStyle RoofMatl Exterior1st Exterior2nd MasVnrType  MasVnrArea ExterQual ExterCond Foundation BsmtQual BsmtCond BsmtExposure BsmtFinType1  BsmtFinSF1 BsmtFinType2  BsmtFinSF2  BsmtUnfSF  TotalBsmtSF Heating HeatingQC CentralAir Electrical  1stFlrSF  2ndFlrSF  LowQualFinSF  GrLivArea  BsmtFullBath  BsmtHalfBath  FullBath  HalfBath  BedroomAbvGr  KitchenAbvGr KitchenQual  TotRmsAbvGrd Functional  Fireplaces FireplaceQu GarageType  GarageYrBlt GarageFinish  GarageCars  GarageArea GarageQual GarageCond PavedDrive  WoodDeckSF  OpenPorchSF  EnclosedPorch  3SsnPorch  ScreenPorch  PoolArea PoolQC Fence MiscFeature  MiscVal  MoSold  YrSold SaleType SaleCondition  SalePrice
0   1          60       RL         65.0     8450   Pave   NaN      Reg         Lvl    AllPub    Inside   

## Data Audit

In [21]:
display(df.info())
display(df.describe(include='all').T)
display(df.isnull().sum().sort_values(ascending=False).head(20))
print('Duplicates:',df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

## Exploratory Data Analysis

In [22]:
sns.histplot(df['SalePrice'],kde=True)
plt.title('SalePrice Distribution')
plt.show()

plt.figure(figsize=(12,10))
sns.heatmap(df.select_dtypes(include=np.number).corr(),cmap='coolwarm')
plt.show()

for col in ['GrLivArea','OverallQual','GarageCars','YearBuilt']:
    sns.scatterplot(data=df,x=col,y='SalePrice')
    plt.show()

sns.boxplot(data=df,x='OverallQual',y='SalePrice')
plt.xticks(rotation=45)
plt.show()


## Train-Test Split

In [23]:
target='SalePrice'
drop_cols=['Id'] if 'Id' in df.columns else []
X=df.drop(columns=[target]+drop_cols)
y=df[target]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=SEED)

num=X.select_dtypes(include=np.number).columns
cat=X.select_dtypes(exclude=np.number).columns


## Preprocessing Pipeline

In [24]:
numeric_pipe=Pipeline([
('imp',SimpleImputer(strategy='median')),
('scale',StandardScaler())
])

categorical_pipe=Pipeline([
('imp',SimpleImputer(strategy='most_frequent')),
('onehot',OneHotEncoder(handle_unknown='ignore'))
])

preprocess=ColumnTransformer([
('num',numeric_pipe,num),
('cat',categorical_pipe,cat)
])


## Dummy Baseline

In [25]:
dummy=DummyRegressor(strategy='mean')
dummy.fit(X_train[num],y_train)
pred=dummy.predict(X_test[num])
mae=mean_absolute_error(y_test,pred)
mse=mean_squared_error(y_test,pred)
rmse=np.sqrt(mse)
r2=r2_score(y_test,pred)
print('Baseline MAE:',mae)
print('Baseline MSE:',mse)
print('Baseline RMSE:',rmse)
print('Baseline R2:',r2)


Baseline MAE: 62575.926451960964
Baseline MSE: 7677095207.783831
Baseline RMSE: 87619.03450611533
Baseline R2: -0.0008824918802490256


## Simple Linear Regression

In [26]:
simple=LinearRegression()
simple.fit(X_train[['GrLivArea']],y_train)
pred=simple.predict(X_test[['GrLivArea']])
print('R2',r2_score(y_test,pred))


R2 0.5542632452871117


## Model Comparison

In [27]:
models={
'Linear Regression':LinearRegression(),
'Ridge':Ridge(alpha=1.0),
'Lasso':Lasso(alpha=0.001,max_iter=20000),
'ElasticNet':ElasticNet(alpha=0.001,l1_ratio=0.5,max_iter=20000),
'Decision Tree':DecisionTreeRegressor(max_depth=6,random_state=SEED),
'Random Forest':RandomForestRegressor(n_estimators=300,min_samples_leaf=2,random_state=SEED,n_jobs=-1),
'Gradient Boosting':GradientBoostingRegressor(random_state=SEED)
}

rows=[]
fitted={}
for name,est in models.items():
    pipe=Pipeline([('preprocess',preprocess),('model',est)])
    pipe.fit(X_train,y_train)
    p=pipe.predict(X_test)
    mae=mean_absolute_error(y_test,p)
    mse=mean_squared_error(y_test,p)
    rmse=np.sqrt(mse)
    r2=r2_score(y_test,p)
    # Adjusted R2 using the number of features after preprocessing
    Xt=pipe.named_steps['preprocess'].transform(X_test)
    n,n_features=Xt.shape
    adj_r2=np.nan if n<=n_features+1 else 1-(1-r2)*(n-1)/(n-n_features-1)
    rows.append({
        'Model':name,
        'MAE':mae,
        'MSE':mse,
        'RMSE':rmse,
        'R2':r2,
        'Adjusted_R2':adj_r2
    })
    fitted[name]=pipe

results_df=pd.DataFrame(rows).sort_values('RMSE')
display(results_df)


               Model           MAE           MSE          RMSE        R2  Adjusted_R2
6  Gradient Boosting  16555.666542  7.024499e+08  26503.771884  0.908420    -3.441639
2              Lasso  18010.801251  8.030381e+08  28337.927127  0.895306    -4.077665
5      Random Forest  17473.413574  8.264305e+08  28747.703974  0.892256    -4.225577
0  Linear Regression  18287.698623  8.687092e+08  29473.873055  0.886744    -4.492908
3         ElasticNet  18809.630842  8.826409e+08  29709.272610  0.884928    -4.580999
1              Ridge  19004.408494  8.905344e+08  29841.823697  0.883899    -4.630911
4      Decision Tree  26729.141432  1.594268e+09  39928.281256  0.792151    -9.080664


## Polynomial Regression

In [28]:
poly=Pipeline([
('imp',SimpleImputer(strategy='median')),
('poly',PolynomialFeatures(degree=2,include_bias=False)),
('scale',StandardScaler()),
('model',Ridge(alpha=1.0))
])
features=['GrLivArea','OverallQual','YearBuilt']
poly.fit(X_train[features],y_train)
print(poly.score(X_test[features],y_test))


0.8064926494606228


## Cross Validation

In [29]:
cv=KFold(n_splits=5,shuffle=True,random_state=SEED)
cv_rows=[]
for name,est in models.items():
    pipe=Pipeline([('preprocess',preprocess),('model',est)])
    scores=cross_validate(pipe,X_train,y_train,cv=cv,
                          scoring=['neg_root_mean_squared_error','r2'])
    cv_rows.append([name,
                    -scores['test_neg_root_mean_squared_error'].mean(),
                    scores['test_r2'].mean()])
cv_df=pd.DataFrame(cv_rows,columns=['Model','CV_RMSE','CV_R2'])
display(cv_df)


               Model       CV_RMSE     CV_R2
0  Linear Regression  36396.352184  0.770001
1              Ridge  32817.038111  0.814688
2              Lasso  36042.143763  0.774688
3         ElasticNet  33588.564083  0.805657
4      Decision Tree  44010.848168  0.671574
5      Random Forest  30637.610510  0.839971
6  Gradient Boosting  27708.844195  0.866805


## Hyperparameter Tuning

In [30]:
grid=GridSearchCV(
Pipeline([('preprocess',preprocess),('model',Ridge())]),
{'model__alpha':[0.001,0.01,0.1,1,10,100]},
cv=5,
scoring='neg_root_mean_squared_error')
grid.fit(X_train,y_train)
print(grid.best_params_)
print(-grid.best_score_)


{'model__alpha': 10}
32577.851183977804


## Residual Analysis

In [31]:
best=fitted[results_df.iloc[0]['Model']]
pred=best.predict(X_test)
res=y_test-pred

plt.scatter(pred,res)
plt.axhline(0,color='red')
plt.xlabel('Predicted')
plt.ylabel('Residual')
plt.show()

sns.histplot(res,kde=True)
plt.show()

plt.scatter(y_test,pred)
plt.plot([y_test.min(),y_test.max()],[y_test.min(),y_test.max()],'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.show()


## Feature Importance

In [32]:
model=best.named_steps['model']
if hasattr(model,'feature_importances_'):
    imp=pd.DataFrame({
        'Feature':best.named_steps['preprocess'].get_feature_names_out(),
        'Importance':model.feature_importances_
    }).sort_values('Importance',ascending=False)
    display(imp.head(20))


                   Feature  Importance
3         num__OverallQual    0.513633
15          num__GrLivArea    0.140511
25         num__GarageCars    0.044125
11        num__TotalBsmtSF    0.042677
8          num__BsmtFinSF1    0.030690
13           num__2ndFlrSF    0.029224
12           num__1stFlrSF    0.027127
2             num__LotArea    0.018103
245  cat__GarageFinish_Unf    0.016698
180       cat__BsmtQual_Ex    0.014974
5           num__YearBuilt    0.014948
23         num__Fireplaces    0.010827
221    cat__KitchenQual_Ex    0.008130
6        num__YearRemodAdd    0.007714
18           num__FullBath    0.007049
4         num__OverallCond    0.006692
49    cat__LandContour_Bnk    0.006381
24        num__GarageYrBlt    0.005140
26         num__GarageArea    0.004230
28        num__OpenPorchSF    0.004101


## Save Outputs

In [34]:
results_df.to_csv('ames_model_comparison.csv',index=False)
cv_df.to_csv('ames_cross_validation.csv',index=False)
joblib.dump(best,'ames_house_price_pipeline.joblib')
